In [83]:
#README


In [84]:
#IMPORTS

import yfinance as yf
import pandas as pd
import numpy as np



In [85]:
#INPUTS

target = yf.Ticker("MSFT")


benchmark = "URTH"
start_date = "2020-12-31"
end_date = "2025-12-31"
save_data = True
period = "5y"
interval = "1wk"
return_calc = "log" #log / simple

peer_group = ["ORCL", "PLTR", "PANW", "CRWD", "FTNT"]


In [86]:
#FUNCTIONS
def get_data(ticker: str, start_date: str, end_date: str, interval: str, save_data: bool):
    data = yf.download(tickers = ticker, start = start_date, end = end_date, interval = interval)
    if save_data:
        path = f"./data/{ticker}_{start_date}_-_{end_date}.csv"
        data.to_csv(path,)
    return data["Close"]

def download_data(tickers: list, start_date: str, end_date:str, interval: str):
    """Download data for the given tickers from Yahoo Finance"""
    data = yf.download(tickers = tickers, start = start_date, end = end_date, interval = interval)
    return data

def save_data(data: pd.DataFrame):
    """Save data to csv file"""
    path = f"./data/{benchmark} + {ticker}_{start_date}_-_{end_date}.csv"
    data.to_csv(path)

def extract_col(data: pd.DataFrame, field: str):
    """Extract columns from the downloaded dataframe"""
    column_data= data[field]
    return column_data

# def quick_beta(peer, benchmark):
#     cov = np.cov(peer, benchmark)[0, 1]
#     var = np.var(benchmark)
#     return cov / var


In [88]:
#CLOSE_DATA_COLLECTION

ticker_package = peer_group + [benchmark]
data_package = download_data(tickers= ticker_package, start_date= start_date, end_date= end_date, interval= interval)
save_data(data_package)
close_data = extract_col(data_package, field= "Close")
close_data.head()


[*********************100%***********************]  6 of 6 completed


Ticker,CRWD,FTNT,ORCL,PANW,PLTR,URTH
Date,,,,,,
2020-12-28,52.955002,29.705999,59.762871,59.231667,23.549999,103.073921
2021-01-04,55.932499,29.628000,58.776691,61.091667,25.200001,105.733055
2021-01-11,54.877499,29.306000,57.292908,60.811668,25.639999,104.119225
2021-01-18,55.880001,30.246000,55.976048,60.770000,32.580002,105.769722
2021-01-25,53.950001,28.950001,56.040966,58.458332,35.180000,102.230309


In [90]:
#BASIC DATA CLEARING

close_data.columns = close_data.columns.get_level_values(0)
close_data = close_data.dropna()
close_data.head()

Ticker,CRWD,FTNT,ORCL,PANW,PLTR,URTH
Date,,,,,,
2020-12-28,52.955002,29.705999,59.762871,59.231667,23.549999,103.073921
2021-01-04,55.932499,29.628000,58.776691,61.091667,25.200001,105.733055
2021-01-11,54.877499,29.306000,57.292908,60.811668,25.639999,104.119225
2021-01-18,55.880001,30.246000,55.976048,60.770000,32.580002,105.769722
2021-01-25,53.950001,28.950001,56.040966,58.458332,35.180000,102.230309


In [91]:
#LOG_RETURN_CALCULATION

log_returns = np.log(close_data / close_data.shift(1))
log_returns = log_returns.dropna()
log_returns.head()

Ticker,CRWD,FTNT,ORCL,PANW,PLTR,URTH
Date,,,,,,
2021-01-04,0.054703,-0.002629,-0.016639,0.030919,0.067718,0.025471
2021-01-11,-0.019042,-0.010928,-0.025569,-0.004594,0.017310,-0.015381
2021-01-18,0.018103,0.031572,-0.023253,-0.000685,0.239545,0.015728
2021-01-25,-0.035149,-0.043794,0.001159,-0.038782,0.076779,-0.034036
2021-02-01,0.035194,0.071509,0.051128,0.079703,-0.032648,0.041378


In [92]:
market = benchmark

raw_betas = {}

for ticker in peer_group:
    cov = np.cov(log_returns[ticker], log_returns[market])[0, 1]
    var = np.var(log_returns[market])
    raw_betas[ticker] = cov / var

raw_betas = pd.Series(raw_betas, name="raw_betas")
raw_betas


ORCL    1.180882
PLTR    2.073678
PANW    1.225052
CRWD    1.685934
FTNT    1.425443
Name: raw_betas, dtype: float64